# Lens A — Funder Power

Who pays for global health research, how concentrated is funding, and does funder nationality shape what gets studied?

### Analyses
1. **HHI by year** — funder concentration trend
2. **Funder × topic cross-tab** — chi-square test of independence
3. **Funder portfolio shift** — each major funder's topic share 2010–2024
4. **Funding nationality bias** — US/UK/EU funder topic profiles
5. **Unfunded paper proportion** — by topic, region, and year

In [ ]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)

DB = 'data/global_health.duckdb'
con = duckdb.connect(DB, read_only=True)

## Load reference data

In [ ]:
# Topic taxonomy for readable labels
topic_labels = pd.read_csv('data/taxonomy/topic_taxonomy.csv')
cat_names = topic_labels.drop_duplicates('category_letter')[['category_letter', 'category_name']]
cat_map = dict(zip(cat_names['category_letter'], cat_names['category_name']))

# Build the core funded-works table:
# Each row = one work-funder pair with canonical funder info joined in
funded_works = con.execute("""
    SELECT w.openalex_id, w.publication_year, w.topic_category,
           w.topic_subtopic, w.method_type, w.study_country,
           f.canonical_name AS funder, f.funder_category, f.funder_country
    FROM works w
    JOIN grants g ON w.openalex_id = g.openalex_id
    JOIN funders f ON REPLACE(g.funder_id, 'https://openalex.org/', '') = f.openalex_id
    WHERE w.topic_category IS NOT NULL
""").fetchdf()

# All works with topic classification (including unfunded)
all_works = con.execute("""
    SELECT openalex_id, publication_year, topic_category,
           topic_subtopic, method_type, study_country
    FROM works
    WHERE topic_category IS NOT NULL
""").fetchdf()

print(f'Total classified works: {len(all_works):,}')
print(f'Funded work–funder pairs: {len(funded_works):,}')
print(f'Unique funders matched: {funded_works["funder"].nunique()}')

---
## A1. HHI by Year — Funder Concentration Trend

The Herfindahl-Hirschman Index measures market concentration. HHI = Σ(share²), where share is each funder's proportion of total funded papers that year. Values range from 0 (perfectly distributed) to 1 (single funder monopoly).

In [ ]:
def compute_hhi(group):
    """Compute HHI from a group of funder-paper pairs."""
    # Count unique papers per funder (a paper with 2 funders counts once each)
    funder_counts = group.groupby('funder')['openalex_id'].nunique()
    total = funder_counts.sum()
    shares = funder_counts / total
    return (shares ** 2).sum()

# HHI per year
hhi_by_year = (
    funded_works
    .groupby('publication_year')
    .apply(compute_hhi, include_groups=False)
    .reset_index()
    .rename(columns={0: 'hhi'})
)

# Also compute top-funder share for context
top_funder_share = []
for year, grp in funded_works.groupby('publication_year'):
    fc = grp.groupby('funder')['openalex_id'].nunique()
    total = fc.sum()
    top = fc.max()
    top_name = fc.idxmax()
    top_funder_share.append({
        'publication_year': year,
        'top_funder': top_name,
        'top_share': top / total,
        'n_funders': len(fc),
        'n_funded_papers': grp['openalex_id'].nunique(),
    })
top_funder_df = pd.DataFrame(top_funder_share)
hhi_df = hhi_by_year.merge(top_funder_df, on='publication_year')

# Plot
fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(hhi_df['publication_year'], hhi_df['hhi'], 'o-', color='#2171b5', linewidth=2)
ax1.set_xlabel('Year')
ax1.set_ylabel('HHI (funder concentration)', color='#2171b5')
ax1.tick_params(axis='y', labelcolor='#2171b5')

ax2 = ax1.twinx()
ax2.bar(hhi_df['publication_year'], hhi_df['n_funders'], alpha=0.3, color='gray', label='Unique funders')
ax2.set_ylabel('Unique funders per year', color='gray')
ax2.tick_params(axis='y', labelcolor='gray')

ax1.set_title('Funder Concentration in Global Health Research (HHI)')
fig.tight_layout()
plt.show()

hhi_df

---
## A2. Funder × Topic Cross-Tabulation + Chi-Square Test

Do funders fund all topics equally, or do certain funders concentrate on specific topics? A chi-square test of independence tells us whether the funder–topic association is statistically significant.

In [ ]:
# Top funders by number of funded papers
top_n = 15
top_funders = (
    funded_works
    .groupby('funder')['openalex_id'].nunique()
    .nlargest(top_n)
    .index.tolist()
)

# Cross-tab: funder × topic category
ct_data = funded_works[funded_works['funder'].isin(top_funders)].copy()
ct_data['topic_name'] = ct_data['topic_category'].map(cat_map)
cross_tab = pd.crosstab(ct_data['funder'], ct_data['topic_name'])

# Chi-square test
chi2, p_value, dof, expected = chi2_contingency(cross_tab)
print(f'Chi-square statistic: {chi2:.1f}')
print(f'p-value: {p_value:.2e}')
print(f'Degrees of freedom: {dof}')
print(f'Result: {"Significant" if p_value < 0.05 else "Not significant"} '
      f'association between funder and topic (α=0.05)')

# Heatmap of proportions (row-normalized: each funder's topic portfolio)
ct_prop = cross_tab.div(cross_tab.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(16, 10))
sns.heatmap(ct_prop, annot=True, fmt='.0%', cmap='YlOrRd', ax=ax,
            linewidths=0.5, linecolor='white', cbar_kws={'label': 'Share of funder portfolio'})
ax.set_title(f'Funder Topic Portfolios (top {top_n} funders, row-normalized)')
ax.set_ylabel('')
ax.set_xlabel('')
plt.xticks(rotation=45, ha='right')
fig.tight_layout()
plt.show()

---
## A3. Funder Portfolio Shift — Topic Share Over Time

How has each major funder's research portfolio changed from 2010 to 2024? This reveals whether funders have diversified or narrowed their focus.

In [ ]:
# Select top 5 funders for clarity
top5 = top_funders[:5]

# Compute topic share per funder per year
portfolio_data = funded_works[funded_works['funder'].isin(top5)].copy()
portfolio_data['topic_name'] = portfolio_data['topic_category'].map(cat_map)

portfolio_shift = (
    portfolio_data
    .groupby(['publication_year', 'funder', 'topic_name'])
    ['openalex_id'].nunique()
    .reset_index(name='n_papers')
)

# Normalize within each funder-year
year_totals = portfolio_shift.groupby(['publication_year', 'funder'])['n_papers'].transform('sum')
portfolio_shift['share'] = portfolio_shift['n_papers'] / year_totals

# For each funder, find their top 5 topics to plot (avoid clutter)
fig, axes = plt.subplots(len(top5), 1, figsize=(14, 4 * len(top5)), sharex=True)
if len(top5) == 1:
    axes = [axes]

for ax, funder in zip(axes, top5):
    funder_data = portfolio_shift[portfolio_shift['funder'] == funder]
    top_topics = (
        funder_data.groupby('topic_name')['n_papers'].sum()
        .nlargest(5).index.tolist()
    )
    for topic in top_topics:
        td = funder_data[funder_data['topic_name'] == topic]
        ax.plot(td['publication_year'], td['share'], 'o-', label=topic, markersize=4)
    ax.set_ylabel('Share')
    ax.set_title(funder, fontsize=11)
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
    ax.set_ylim(0, None)

axes[-1].set_xlabel('Year')
fig.suptitle('Funder Portfolio Shift — Topic Shares Over Time', fontsize=14, y=1.01)
fig.tight_layout()
plt.show()

---
## A4. Funding Nationality Bias — US / UK / EU Funder Topic Profiles

Do US, UK, and EU-based funders prioritize different topics? Comparing their portfolio distributions reveals potential geographic biases in what gets funded.

In [ ]:
# Group funders by nationality
eu_countries = ['Multilateral', 'DE', 'FR', 'NL', 'SE', 'NO', 'DK', 'BE', 'CH', 'IT', 'ES']

def classify_nationality(country):
    if country == 'US':
        return 'US'
    elif country == 'UK':
        return 'UK'
    elif country in eu_countries:
        return 'EU / Multilateral'
    else:
        return 'Other'

nationality_data = funded_works.copy()
nationality_data['funder_nationality'] = nationality_data['funder_country'].apply(classify_nationality)
nationality_data['topic_name'] = nationality_data['topic_category'].map(cat_map)

# Cross-tab: nationality × topic
nat_ct = pd.crosstab(nationality_data['funder_nationality'], nationality_data['topic_name'])
nat_prop = nat_ct.div(nat_ct.sum(axis=1), axis=0)

# Plot grouped horizontal bar chart
fig, ax = plt.subplots(figsize=(14, 8))
nat_plot = nat_prop.loc[['US', 'UK', 'EU / Multilateral']].T.sort_index()
nat_plot.plot(kind='barh', ax=ax, width=0.8)
ax.set_xlabel('Share of funded papers')
ax.set_title('Topic Funding Profiles by Funder Nationality')
ax.legend(title='Funder nationality')
fig.tight_layout()
plt.show()

---
## A5. Unfunded Paper Proportion — By Topic, Region, and Year

What share of published research has no identifiable funder? This varies dramatically by topic and geography, revealing where research happens without formal funding support.

In [ ]:
# Identify which works have at least one matched funder
funded_ids = set(funded_works['openalex_id'].unique())
all_works['is_funded'] = all_works['openalex_id'].isin(funded_ids)
all_works['topic_name'] = all_works['topic_category'].map(cat_map)

# Unfunded rate by topic
unfunded_by_topic = (
    all_works.groupby('topic_name')
    .agg(total=('openalex_id', 'count'), funded=('is_funded', 'sum'))
    .assign(unfunded_rate=lambda x: 1 - x['funded'] / x['total'])
    .sort_values('unfunded_rate', ascending=True)
)

fig, ax = plt.subplots(figsize=(12, 8))
unfunded_by_topic['unfunded_rate'].plot(kind='barh', ax=ax, color='#e6550d')
ax.set_xlabel('Unfunded paper proportion')
ax.set_title('Proportion of Papers Without Identified Funding, by Topic')
ax.axvline(x=unfunded_by_topic['unfunded_rate'].mean(), color='gray',
           linestyle='--', label=f'Mean: {unfunded_by_topic["unfunded_rate"].mean():.0%}')
ax.legend()
for i, (idx, row) in enumerate(unfunded_by_topic.iterrows()):
    ax.text(row['unfunded_rate'] + 0.01, i, f'{row["unfunded_rate"]:.0%}', va='center', fontsize=9)
fig.tight_layout()
plt.show()

# Unfunded rate by year
unfunded_by_year = (
    all_works.groupby('publication_year')
    .agg(total=('openalex_id', 'count'), funded=('is_funded', 'sum'))
    .assign(unfunded_rate=lambda x: 1 - x['funded'] / x['total'])
)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(unfunded_by_year.index, unfunded_by_year['unfunded_rate'], 'o-', color='#e6550d', linewidth=2)
ax.set_xlabel('Year')
ax.set_ylabel('Unfunded proportion')
ax.set_title('Proportion of Papers Without Identified Funding Over Time')
ax.set_ylim(0, 1)
fig.tight_layout()
plt.show()

---
## Save Results to DuckDB

Store computed summaries so the findings writer and visualization agents can access them.

In [ ]:
con.close()

# Reopen with write access to save results
con_write = duckdb.connect(DB)

# HHI concentration trend
con_write.execute('CREATE OR REPLACE TABLE funder_hhi_by_year AS SELECT * FROM ?', [hhi_df])

# Top funder summary with paper counts
top_funders_summary = (
    funded_works
    .groupby(['funder', 'funder_category', 'funder_country'])
    ['openalex_id'].nunique()
    .reset_index(name='n_papers')
    .sort_values('n_papers', ascending=False)
)
con_write.execute('CREATE OR REPLACE TABLE funder_summary AS SELECT * FROM ?', [top_funders_summary])

# Funder × topic cross-tab (counts)
funder_topic_summary = (
    funded_works
    .groupby(['funder', 'funder_category', 'funder_country',
              'topic_category', 'publication_year'])
    ['openalex_id'].nunique()
    .reset_index(name='n_papers')
)
con_write.execute('CREATE OR REPLACE TABLE funder_topic_summary AS SELECT * FROM ?', [funder_topic_summary])

con_write.close()
print('Lens A results saved to DuckDB.')